In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [11]:
torch.manual_seed(1)

### Discrete Probability Distribution "이산 확률 분포"

1. 확률 분포 함수 PDF함수, 면적 == 확률 (px=x) ??알수 없다. 
2. 이산 확률 분포, 점 == 확률, "주사위 6이 나올 확률, 면적이 아니다!"

ex. 
1. p(주먹:가위) = ?
2. p(가위:가위) = ? 
3. p(보:가위) = ?

## softmax 
- Convert numbers to probabilities with softmax

In [10]:
z = torch.FloatTensor([1, 2, 3]) # max = (0, 0, 1) 소프트 맥스는 이 값을 부드럽게, 출력 (hypothesis참고) 
hypothesis = F.softmax(z, dim=0)
print(hypothesis)

tensor([0.0900, 0.2447, 0.6652])


In [6]:
hypothesis.sum()

tensor(1.)

### CrossEntropy - OneHot벡터의 Index

In [14]:
z = torch.rand(3, 5, requires_grad=True) # z = rand 3, 5라는 gradient를 배운다. (3,5)
hypothesis = F.softmax(z, dim=1) # z를 dimension 1에 대해서 0.243, 0.1624, 0.2322....에 대해서 수행. 
print(hypothesis)

tensor([[0.2645, 0.1639, 0.1855, 0.2585, 0.1277],
        [0.2430, 0.1624, 0.2322, 0.1930, 0.1694],
        [0.2226, 0.1986, 0.2326, 0.1594, 0.1868]], grad_fn=<SoftmaxBackward>)


In [15]:
y = torch.randint(5, (3,)).long()  # class개수 5개, sample수 3개. 
print(y)

tensor([0, 2, 1])


### CrossEntropy - OneHot벡터

In [17]:
y_one_hot = torch.zeros_like(hypothesis) # y_one_hot = (3,5)
y_one_hot.scatter_(1, y.unsqueeze(1), 1) # dimension 1에 대해서, y = (3,1) y_unsqueeze = (3,1) # y=[0,2,1]을 세로로 

tensor([[1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0.]])

In [18]:
cost = (y_one_hot * -torch.log(hypothesis)).sum(dim=1).mean()
print(cost)

tensor(1.4689, grad_fn=<MeanBackward0>)


<font color = 'red'> Cross-Entropy Loss with torch.nn.functional

In [21]:
# Low Level
a = torch.log(F.softmax(z, dim=1))
#High Level
b = F.log_softmax(z, dim=1)
print(a)
print(b)

tensor([[-1.3301, -1.8084, -1.6846, -1.3530, -2.0584],
        [-1.4147, -1.8174, -1.4602, -1.6450, -1.7758],
        [-1.5025, -1.6165, -1.4586, -1.8360, -1.6776]], grad_fn=<LogBackward>)
tensor([[-1.3301, -1.8084, -1.6846, -1.3530, -2.0584],
        [-1.4147, -1.8174, -1.4602, -1.6450, -1.7758],
        [-1.5025, -1.6165, -1.4586, -1.8360, -1.6776]],
       grad_fn=<LogSoftmaxBackward>)


<font color = 'red'> Cross-Eentropy loss, one-hot vector

In [26]:
# Low Level
a = (y_one_hot * -torch.log(F.softmax(z, dim=1))).sum(dim=1).mean()
# High Level
b = F.nll_loss(F.log_softmax(z, dim=1), y)
print(a)
print(b)
F.cross_entropy(z, y)

tensor(1.4689, grad_fn=<MeanBackward0>)
tensor(1.4689, grad_fn=<NllLossBackward>)


tensor(1.4689, grad_fn=<NllLossBackward>)

In [32]:
x_train = [[1, 2, 1, 1],
           [2, 1, 3, 2],
           [3, 1, 3, 4],
           [4, 1, 5, 5],
           [1, 7, 5, 5],
           [1, 2, 5, 6],
           [1, 6, 6, 6],
           [1, 7, 7, 7]]
y_train = [2, 2, 2, 1, 1, 1, 0, 0]
x_train = torch.FloatTensor(x_train)
y_train = torch.LongTensor(y_train)
# x_train = (8, 4), y_train = (8, )

In [39]:
y_train.shape

torch.Size([8])

In [43]:
#모델 초기화
W = torch.zeros((4, 3), requires_grad = True) # samples = 8, classes = 3, 입력vector dim = 4
b = torch.zeros(1, requires_grad = True)

#Optimizer 설정
optimizer = optim.SGD([W, b], lr=0.1)

nb_epochs = 1000
for epoch in range(nb_epochs + 1):
    #cost 계산
    hypothesis = F.softmax(x_train.matmul(W) + b, dim=1)
    y_one_hot = torch.zeros_like(hypothesis) #one-hot 벡터
    y_one_hot.scatter_(1, y_train.unsqueeze(1), 1) 
    cost = (y_one_hot * -torch.log(F.softmax(hypothesis, dim=1))).sum(dim=1)
    
    #cost로 H(x)계산
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()
    
    if epoch % 100 == 0:
        print('Epoch {:4d}/{} Cost: {:.6f}'.format(epoch, nb_epochs, cost.item()))

IndexError: tuple index out of range

In [36]:
#모델 초기화
W = torch.zeros((4, 3), requires_grad = True) # samples = m, classes = 3, 입력vector dim = 4
b = torch.zeros(1, requires_grad = True)

#Optimizer 설정
optimizer = optim.SGD([W, b], lr=0.1)

nb_epochs = 1000
for epoch in range(nb_epochs + 1):
    #cost 계산
    hypothesis = F.softmax(x_train.matmul(W) + b, dim=1)
    cost = F.cross_entropy(z, y_train)
    
    #cost로 H(x)계산
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()
    
    if epoch % 100 == 0:
        print('Epoch {:4d}/{} Cost: {:.6f}'.format(epoch, nb_epochs, cost.item()))

ValueError: Expected input batch_size (3) to match target batch_size (8).

### High-level Implementation with nn.Module

In [45]:
class SoftmaxClassifierModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 3)
    def forward(self, x):
        return self.linear(x) # (4, 3) --> (3, 4)

In [46]:
model = SoftmaxClassifierModel()

In [47]:
optimizer = optim.SGD(model.parameters(), lr=0.1)

nb_epochs = 1000
for epoch in range(nb_epochs + 1):
    
    # H(x) 계산
    prediction = model(x_train)
    # Cost 계산
    cost = F.cross_entropy(prediction, y_train)
    
    # Cost로 H(x) 개선
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()
    
    if epoch % 100 == 0:
        print('Epoch {:4d}/{} Cost: {:.6f}'.format(epoch, nb_epochs, cost.item()))

Epoch    0/1000 Cost: 1.849513
Epoch  100/1000 Cost: 0.689894
Epoch  200/1000 Cost: 0.609258
Epoch  300/1000 Cost: 0.551218
Epoch  400/1000 Cost: 0.500141
Epoch  500/1000 Cost: 0.451947
Epoch  600/1000 Cost: 0.405051
Epoch  700/1000 Cost: 0.358733
Epoch  800/1000 Cost: 0.312911
Epoch  900/1000 Cost: 0.269522
Epoch 1000/1000 Cost: 0.241921
